# 🔗 Embedding Merger

**Amaç:** Patch halindeki embedding dosyalarını birleştir

**Giriş:**
- `furniture-retrieval/embeddings/` altındaki `*_patch_*.csv` ve `*_patch_*.npy` dosyaları

**Çıkış:**
- `all_embeddings.parquet`: Her satırda `folder_name`, `image_name`, `embedding` (512-dim liste)

In [1]:
# 🔗 Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 📦 Gerekli paketleri yükle
!pip install -q pandas pyarrow tqdm

In [3]:
# ⚙️ AYARLAR
import os
from pathlib import Path

# Embedding'lerin bulunduğu klasör (Drive)
EMBEDDINGS_DIR = "/content/drive/MyDrive/furniture-retrieval/embeddings"

# Çıktı dosyası (aynı klasöre kaydedilecek)
OUTPUT_FILE = os.path.join(EMBEDDINGS_DIR, "all_embeddings.parquet")

print(f"📁 Embeddings: {EMBEDDINGS_DIR}")
print(f"📁 Çıktı: {OUTPUT_FILE}")

📁 Embeddings: /content/drive/MyDrive/furniture-retrieval/embeddings
📁 Çıktı: /content/drive/MyDrive/furniture-retrieval/embeddings/all_embeddings.parquet


In [4]:
# 📊 Mevcut patch dosyalarını kontrol et
embeddings_path = Path(EMBEDDINGS_DIR)

csv_files = sorted(embeddings_path.glob("*_patch_*.csv"))
npy_files = sorted(embeddings_path.glob("*_patch_*.npy"))

print(f"📊 Bulunan dosyalar:")
print(f"   CSV: {len(csv_files)}")
print(f"   NPY: {len(npy_files)}")

# Toplam boyut
total_size = sum(f.stat().st_size for f in npy_files)
print(f"   NPY toplam boyut: {total_size / 1e9:.2f} GB")

# Klasörleri listele
folders = set()
for f in csv_files:
    # Format: foldername_patch_N.csv
    folder_name = "_".join(f.stem.split("_")[:-2])  # patch_N kısmını çıkar
    folders.add(folder_name)

print(f"\n📁 Klasörler ({len(folders)}):")
for folder in sorted(folders):
    count = len([f for f in csv_files if f.stem.startswith(folder + "_patch_")])
    print(f"   {folder}: {count} patch")

📊 Bulunan dosyalar:
   CSV: 944
   NPY: 944
   NPY toplam boyut: 3.79 GB

📁 Klasörler (42):
   baby_kids: 17 patch
   backyard: 38 patch
   basement: 30 patch
   bathroom: 39 patch
   bedroom: 39 patch
   breakfast_nook: 1 patch
   craft: 2 patch
   deck: 32 patch
   deck_railing: 3 patch
   dining_room: 39 patch
   driveway: 8 patch
   exterior: 38 patch
   family_room: 39 patch
   firepit: 25 patch
   fireplace: 2 patch
   front_yard: 33 patch
   game_room: 5 patch
   garage: 14 patch
   great_room: 22 patch
   gym: 8 patch
   home_bar: 37 patch
   home_office: 31 patch
   kitchen: 39 patch
   kitchen_backsplash: 2 patch
... [3 lines truncated for the showcase repo] ...
   living_room: 39 patch
   mudroom: 6 patch
   outdoor_kitchen: 21 patch
   pantry: 18 patch
   patio: 39 patch
   pergola: 16 patch
   pool: 39 patch
   poolhouse: 5 patch
   porch: 36 patch
   powder_room: 39 patch
   privacy_fence: 2 patch
   small_closet: 2 patch
   storage_closet: 30 patch
   sunroom: 25 patch
 

---
## 🔗 Patch'leri Birleştir

Tüm patch dosyalarını tek bir parquet dosyasında birleştir.

**Çıktı formatı:**
| folder_name | image_name | embedding |
|-------------|------------|-----------|
| wine_cellar | <image>.jpg | [0.012, -0.034, ...] |

In [5]:
# 🔗 Merge fonksiyonu
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import json

def merge_patches_to_parquet(embeddings_dir: str, output_file: str):
    """
    Tüm patch CSV ve NPY dosyalarını birleştir.

    Çıktı parquet formatı:
      - folder_name: Klasör adı (wine_cellar, kitchen, vb.)
      - image_name: Resim dosya adı (<image>.jpg)
      - embedding: 512 boyutlu embedding listesi
    """
    embeddings_path = Path(embeddings_dir)

    # Tüm CSV dosyalarını bul
    csv_files = sorted(embeddings_path.glob("*_patch_*.csv"))

    if not csv_files:
        print("⚠️ Patch dosyası bulunamadı!")
        return None

    print(f"📦 {len(csv_files)} patch dosyası bulundu")

    all_rows = []
    total_images = 0
    skipped_patches = 0

    for csv_file in tqdm(csv_files, desc="Patch'ler işleniyor"):
        npy_file = csv_file.with_suffix('.npy')

        if not npy_file.exists():
            print(f"⚠️ NPY bulunamadı: {npy_file.name}")
            skipped_patches += 1
            continue

        # CSV yükle
        df = pd.read_csv(csv_file)

        # NPY yükle
        embeddings = np.load(npy_file)

        # Boyut kontrolü
        if len(df) != len(embeddings):
            print(f"⚠️ Boyut uyuşmazlığı: {csv_file.name} ({len(df)} vs {len(embeddings)})")
            skipped_patches += 1
            continue

        # Her satır için folder_name, image_name ve embedding çıkar
        for idx, row in df.iterrows():
            # image_path: /content/data/images/wine_cellar/<image>.jpg
            image_path = row['image_path']
            folder_name = row['folder']

            # Sadece dosya adını al
            image_name = Path(image_path).name

            # Embedding'i listeye çevir (parquet için)
            embedding = embeddings[idx].tolist()

            all_rows.append({
                'folder_name': folder_name,
                'image_name': image_name,
                'embedding': embedding
            })

        total_images += len(df)

    print(f"\n📊 Birleştirme sonucu:")
    print(f"   İşlenen patch: {len(csv_files) - skipped_patches}")
    print(f"   Atlanan patch: {skipped_patches}")
    print(f"   Toplam resim: {total_images}")

    # DataFrame oluştur
    print(f"\n💾 Parquet dosyası oluşturuluyor...")
    result_df = pd.DataFrame(all_rows)

    # Parquet olarak kaydet
    result_df.to_parquet(output_file, index=False)

    file_size = Path(output_file).stat().st_size / 1e9
    print(f"   ✅ Kaydedildi: {output_file}")
    print(f"   📦 Dosya boyutu: {file_size:.2f} GB")

    # Klasör dağılımı
    print(f"\n📁 Klasör dağılımı:")
    folder_counts = result_df['folder_name'].value_counts()
    for folder, count in sorted(folder_counts.items()):
        print(f"   {folder}: {count}")

    # Summary JSON kaydet (dataset_creator uyumu için)
    summary = {
        "total_images": len(result_df),
        "embedding_dim": 512,
        "format": "parquet",
        "columns": ["folder_name", "image_name", "embedding"],
        "folders": folder_counts.to_dict()
    }

    summary_file = Path(embeddings_dir) / "summary.json"
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"   ✅ {summary_file.name}")

    return result_df

print("✅ Merge fonksiyonu yüklendi")

✅ Merge fonksiyonu yüklendi


In [6]:
# ▶️ BİRLEŞTİR
result_df = merge_patches_to_parquet(EMBEDDINGS_DIR, OUTPUT_FILE)

📦 944 patch dosyası bulundu


Patch'ler işleniyor:   0%|          | 0/944 [00:00<?, ?it/s]


📊 Birleştirme sonucu:
   İşlenen patch: 944
   Atlanan patch: 0
   Toplam resim: 1850924

💾 Parquet dosyası oluşturuluyor...
   ✅ Kaydedildi: /content/drive/MyDrive/furniture-retrieval/embeddings/all_embeddings.parquet
   📦 Dosya boyutu: 5.89 GB

📁 Klasör dağılımı:
   baby_kids: 33539
   backyard: 74059
   basement: 58787
   bathroom: 77921
   bedroom: 77775
   breakfast_nook: 1993
   craft: 2133
   deck: 62294
   deck_railing: 4214
   dining_room: 77590
   driveway: 15409
   exterior: 74656
   family_room: 77448
   firepit: 49843
   fireplace: 3955
   front_yard: 64491
   game_room: 8795
   garage: 27370
   great_room: 43099
... [9 lines truncated for the showcase repo] ...
   mudroom: 10390
   outdoor_kitchen: 40342
   pantry: 34398
   patio: 77730
   pergola: 31071
   pool: 77451
   poolhouse: 9052
   porch: 71546
   powder_room: 77430
   privacy_fence: 2076
   small_closet: 3899
   storage_closet: 59359
   sunroom: 48045
   wine_cellar: 48176
   ✅ summary.json


---
## 🔍 Sonucu Kontrol Et

In [6]:
# 🔍 Parquet dosyasını kontrol et
output_path = Path(OUTPUT_FILE)

if output_path.exists():
    # Dosya bilgisi
    size_gb = output_path.stat().st_size / 1e9
    print(f"📁 Dosya: {output_path.name}")
    print(f"📦 Boyut: {size_gb:.2f} GB")

    # İlk birkaç satırı oku
    df = pd.read_parquet(OUTPUT_FILE)
    print(f"📊 Toplam satır: {len(df):,}")
    print(f"📊 Kolonlar: {list(df.columns)}")

    print(f"\n🔍 İlk 5 satır:")
    for idx, row in df.head(5).iterrows():
        emb_preview = row['embedding'][:5]  # İlk 5 değer
        print(f"   {row['folder_name']}/{row['image_name'][:30]}... -> [{emb_preview[0]:.4f}, ...]")
else:
    print("⚠️ Dosya bulunamadı!")

📁 Dosya: all_embeddings.parquet
📦 Boyut: 5.89 GB
📊 Toplam satır: 1,850,924
📊 Kolonlar: ['folder_name', 'image_name', 'embedding']

🔍 İlk 5 satır:
   baby_kids/<image>.jpg -> [-0.0163, ...]
   baby_kids/<image>.jpg -> [0.0043, ...]
   baby_kids/<image>.jpg -> [0.0385, ...]
   baby_kids/<image>.jpg -> [-0.0113, ...]
   baby_kids/<image>.jpg -> [0.0335, ...]


In [7]:
# 📊 Belirli bir image'ın embedding'ini bul
def find_embedding(df, folder_name: str = None, image_name: str = None):
    """
    Belirli bir image'ın embedding'ini bul.

    Kullanım:
        find_embedding(df, folder_name="wine_cellar", image_name="ff1157300133a5f8")
    """
    query = df

    if folder_name:
        query = query[query['folder_name'] == folder_name]

    if image_name:
        # Partial match için contains kullan
        query = query[query['image_name'].str.contains(image_name, case=False)]

    if len(query) == 0:
        print("⚠️ Sonuç bulunamadı")
        return None

    print(f"📊 {len(query)} sonuç bulundu:")
    for idx, row in query.head(5).iterrows():
        emb = np.array(row['embedding'])
        print(f"\n   📁 {row['folder_name']}/{row['image_name']}")
        print(f"   📐 Embedding shape: {emb.shape}")
        print(f"   🔢 İlk 10 değer: {emb[:10].round(4).tolist()}")

    return query

# Örnek kullanım
print("💡 Kullanım: find_embedding(df, folder_name='wine_cellar', image_name='ff1157')")

💡 Kullanım: find_embedding(df, folder_name='wine_cellar', image_name='ff1157')


---
## 📥 Dataset Creator için Hazırlık

Parquet dosyasını dataset_creator ile kullanmak için:

```python
# dataset_creator.py güncellenmeli veya yeni bir script yazılmalı
# Parquet formatını destekleyecek şekilde

# Örnek kullanım:
import pandas as pd
import numpy as np

df = pd.read_parquet("all_embeddings.parquet")
embeddings = np.array(df['embedding'].tolist())  # (N, 512)
metadata = df[['folder_name', 'image_name']]

# Artık greedy selection yapabilirsin
```